# שבוע 04: טעינת נקודות ציון והמחשה גרפית

בשיעור זה נלמד כיצד לטעון קובץ TPS המכיל קואורדינטות של נקודות ציון ארכיאולוגיות,  
ונמחיש אותן באמצעות גרפים. נשתמש במטבעות רומיים (דנאריי) כמקרה בוחן.

**מטרות השיעור:**
- טעינת קובץ TPS מ-GitHub
- הבנת מבנה הנתונים של נקודות ציון
- יצירת ויזואליזציות בסיסיות

In [ ]:
!pip install python-bidi -q

import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
rtl = get_display
print('הכל מוכן!')

## פונקציית ניתוח קבצי TPS

קבצי TPS מכילים נקודות ציון בפורמט טקסט.  
שימו לב: קבצים אירופיים משתמשים בפסיק כנקודה עשרונית — הפונקציה מטפלת בכך אוטומטית.

In [ ]:
def parse_tps(filepath_or_text):
    """Works with both local path string and text content."""
    if '\n' in filepath_or_text:
        lines = filepath_or_text.strip().split('\n')
    else:
        with open(filepath_or_text, encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
    specimens, ids = [], []
    i = 0
    while i < len(lines):
        line = lines[i].strip() if hasattr(lines[i], 'strip') else lines[i]
        if line.startswith('LM='):
            n_lm = int(line.split('=')[1])
            coords = []
            for j in range(n_lm):
                i += 1
                parts = lines[i].strip().replace(',', '.').split()
                coords.append([float(parts[0]), float(parts[1])])
            specimens.append(np.array(coords))
        elif line.startswith('ID='):
            ids.append(line.split('=')[1].strip())
        i += 1
    return np.array(specimens), ids

print('parse_tps מוכן')

## טעינת נתוני המטבעות

נטען את קובץ ה-TPS של מטבעות הדריאנוס מ-GitHub.  
אם הטעינה נכשלת, ניצור נתונים סינתטיים לצורך הדגמה.

In [ ]:
import urllib.request

HADRIAN_URL = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/coins/hadrian.tps'

try:
    with urllib.request.urlopen(HADRIAN_URL, timeout=15) as r:
        text = r.read().decode('utf-8', errors='replace')
    specimens, ids = parse_tps(text)
    print(f'נטענו {len(specimens)} מטבעות, {specimens.shape[1]} נקודות ציון כל אחד')
    DATA_SOURCE = 'GitHub'
except Exception as e:
    print(f'שגיאה בטעינה: {e}')
    print('משתמשים בנתונים סינתטיים...')
    np.random.seed(42)
    n_coins, n_lm = 20, 16
    angles = np.linspace(0, 2*np.pi, n_lm, endpoint=False)
    base = np.column_stack([np.cos(angles)*100, np.sin(angles)*80])
    specimens = np.array([base + np.random.randn(n_lm, 2)*5 for _ in range(n_coins)])
    ids = [f'Hadrian_{i+1}' for i in range(n_coins)]
    DATA_SOURCE = 'Synthetic'

print(f'מקור נתונים: {DATA_SOURCE}')
print(f'צורת המערך: {specimens.shape}  (דגימות × נקודות × קואורדינטות)')

## ויזואליזציה 1: גריד של 6 מטבעות

נציג את 6 המטבעות הראשונים עם נקודות הציון ממוספרות.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for idx in range(6):
    ax = axes[idx]
    lm = specimens[idx]
    ax.scatter(lm[:, 0], lm[:, 1], c='steelblue', s=60, zorder=3)
    ax.plot(np.append(lm[:, 0], lm[0, 0]),
            np.append(lm[:, 1], lm[0, 1]),
            'gray', lw=0.8, alpha=0.6)
    for j, (x, y) in enumerate(lm):
        ax.text(x, y, str(j+1), fontsize=6, ha='center', va='bottom', color='darkred')
    label = ids[idx] if idx < len(ids) else f'דגימה {idx+1}'
    ax.set_title(label, fontsize=9)
    ax.set_aspect('equal')
    ax.axis('off')

fig.suptitle(rtl('נקודות ציון: 6 מטבעות הדריאנוס הראשונים'), fontsize=14)
plt.tight_layout()
plt.show()

## ויזואליזציה 2: שכבת-על של כל הדגימות

כאשר מציגים את כל המטבעות יחד (לפני יישור GPA),  
רואים שהם נמצאים במיקומים, גדלים וזוויות שונות.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(specimens)))

for i, lm in enumerate(specimens):
    ax.scatter(lm[:, 0], lm[:, 1], color=colors[i], s=15, alpha=0.7)
    ax.plot(np.append(lm[:, 0], lm[0, 0]),
            np.append(lm[:, 1], lm[0, 1]),
            color=colors[i], lw=0.5, alpha=0.4)

ax.set_title(rtl('שכבת-על: כל המטבעות לפני יישור (GPA)'), fontsize=13)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_aspect('equal')
sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(1, len(specimens)))
sm.set_array([])
plt.colorbar(sm, ax=ax, label='מספר מטבע')
plt.tight_layout()
plt.show()

print(f'סה"כ {len(specimens)} מטבעות מוצגים')
print(f'טווח X: {specimens[:,:,0].min():.1f} – {specimens[:,:,0].max():.1f}')
print(f'טווח Y: {specimens[:,:,1].min():.1f} – {specimens[:,:,1].max():.1f}')

In [ ]:
def centroid_size(lm):
    c = lm.mean(axis=0)
    return np.sqrt(np.sum((lm - c)**2))

sizes = np.array([centroid_size(s) for s in specimens])
print(f'גודל קנטרואיד ממוצע: {sizes.mean():.2f}')
print(f'סטיית תקן: {sizes.std():.2f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(sizes, bins=8, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(sizes.mean(), color='red', lw=2, linestyle='--', label=rtl('ממוצע'))
ax.set_xlabel(rtl('גודל קנטרואיד'), fontsize=11)
ax.set_ylabel(rtl('תדירות'), fontsize=11)
ax.set_title(rtl('התפלגות גדלי קנטרואיד — מטבעות הדריאנוס'), fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

## סיכום

בשיעור זה למדנו:
- **פורמט TPS**: כל דגימה מתחילה ב-`LM=N`, ואחריה N שורות קואורדינטות X Y
- **טיפול בנתונים אירופיים**: החלפת פסיקים בנקודות עשרוניות
- **ויזואליזציה**: גריד לדגימות בודדות ושכבת-על לכלל המדגם
- **גודל קנטרואיד**: מדד גודל בסיסי לנקודות ציון

**שאלות לחשיבה:**
1. מדוע יש שונות בין המטבעות בגרף שכבת-העל?
2. כיצד גודל הקנטרואיד שונה ממדידת קוטר המטבע?
3. מה יקרה לגרף שכבת-העל לאחר GPA?